# Named Entity Recognition


Named entity recognition (NER) is a technique in natural language processing (NLP) that aims to identify and classify named entities in unstructured text. Named entities are words or phrases that refer to specific entities, such as persons, organizations, locations, dates, numbers, etc. For example, in the sentence “Barack Obama was born in Hawaii on August 4, 1961”, the named entities are “Barack Obama” (person), “Hawaii” (location), and “August 4, 1961” (date).

NER is an important task for many NLP applications, such as information extraction, question answering, text summarization, and knowledge graph construction. By recognizing and categorizing named entities, NER can help machines understand the meaning and context of natural language texts, and enable various downstream analyses and applications.

There are different methods and approaches for performing NER, such as rule-based, dictionary-based, machine learning-based, and deep learning-based. Each method has its own advantages and challenges, depending on the domain, language, and complexity of the text. <img title="NER" alt="example of NER" src="https://miro.medium.com/v2/resize:fit:720/format:webp/0*zo068pv-Sn6UjsSW">



## installing needed libraries

In [ ]:
!pip install transformers


## importing libraries

In [ ]:
import torch
from transformers import AutoModelForTokenClassification, AutoTokenizer
from transformers import pipeline
import pandas as pd
from collections import Counter
import pandas as pd


In [ ]:

df = pd.read_csv("/content/drive/MyDrive/NLP/NLU/NLU/data/NER/names.csv", header=None)

In [ ]:
# NUM  = 500
label_convert = {

    "B-LOC":"loc", "I-LOC":"loc","I-PER":"per", "B-PER":"per",
    "B-MISC":"O", "I-MISC":"O", "O":"O", "B-ORG":"O","I-ORG":"O",
}
# MODEL = "abdusah/arabert-ner"
# MODEL = "Hatman/bert-finetuned-ner"
MODEL = "FacebookAI/xlm-roberta-large-finetuned-conll03-english"

## loading pretrained model

In [ ]:

# Load the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForTokenClassification.from_pretrained(MODEL)

nlp = pipeline("ner", model=model, tokenizer=tokenizer, device=device)



tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/852 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Some weights of the model checkpoint at FacebookAI/xlm-roberta-large-finetuned-conll03-english were not used when initializing XLMRobertaForTokenClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


## helper function

In [ ]:
def split_text(text):     return text.split()

def convert_entities(entities):
  entities_map = {"per":"PERSON", "org": "ORGANIZATION", "loc":"LOCATION"}

  mapped_entities = {}
  current_entity = None
  start_index = None

  for i, entity_tag in enumerate(entities):
      if entity_tag != "O" and entity_tag != current_entity:
          # New entity starts
          if current_entity:mapped_entities.setdefault(entities_map[current_entity], []).append({"start": start_index, "end": i - 1})
          current_entity = entity_tag
          start_index = i
      elif entity_tag == "O" and current_entity:
          # Previous entity ends
          mapped_entities.setdefault(entities_map[current_entity], []).append({"start": start_index, "end": i - 1})
          current_entity = None;start_index = None
  return mapped_entities

## processing function

In [ ]:
def start_porcess(sentence):
  text = split_text(sentence);  annotations = nlp(text);  lines, entities = [], []
  for idx,  sentence in enumerate(annotations):
    if sentence == []:  lines.append(text[idx]); entities.append(label_convert["O"])
    else :
      entitie = [label_convert[temp["entity"]] for temp in sentence]
      entity_counts = Counter(entitie)
      most_repeated_entity = entity_counts.most_common(1)[0][0]
      lines.append(text[idx]); entities.append(most_repeated_entity)


  results =  {
      "text":" ".join(lines),
      "entities": convert_entities(entities)
  }
  return results

In [ ]:
start_porcess(
    'الصالحية المفرق غيث الطراونة أمر جلالة الملك عبدالله الثاني أمس بتنفيذ حزمة من المشاريع التعليمية والصحية والتنموية \
    وأخرى مرتبطة بالأندية الشبابية و 27 وحدة سكنية في قضاء الصالحية ونايفة في البادية الشرقية خلال ستة اشهر بتمويل من الديوان الملكي الهاشمي'
    )

{'text': 'الصالحية المفرق غيث الطراونة أمر جلالة الملك عبدالله الثاني أمس بتنفيذ حزمة من المشاريع التعليمية والصحية والتنموية وأخرى مرتبطة بالأندية الشبابية و 27 وحدة سكنية في قضاء الصالحية ونايفة في البادية الشرقية خلال ستة اشهر بتمويل من الديوان الملكي الهاشمي',
 'entities': {'ORGANIZATION': [{'start': 1, 'end': 1},
   {'start': 3, 'end': 3},
   {'start': 5, 'end': 5},
   {'start': 7, 'end': 7},
   {'start': 13, 'end': 14},
   {'start': 18, 'end': 18},
   {'start': 20, 'end': 20},
   {'start': 30, 'end': 30},
   {'start': 32, 'end': 32},
   {'start': 37, 'end': 37}],
  'LOCATION': [{'start': 2, 'end': 2},
   {'start': 4, 'end': 4},
   {'start': 6, 'end': 6},
   {'start': 11, 'end': 11},
   {'start': 17, 'end': 17},
   {'start': 24, 'end': 24},
   {'start': 26, 'end': 26},
   {'start': 31, 'end': 31},
   {'start': 34, 'end': 34},
   {'start': 38, 'end': 38}]}}

In [ ]:

for text in df[0]:
  print(start_porcess(text))

{'text': 'حسن معاك', 'entities': {'PERSON': [{'start': 0, 'end': 0}]}}
{'text': 'دكتور سالم سعيد محمد سالم', 'entities': {'PERSON': [{'start': 2, 'end': 3}]}}
{'text': 'سناء ابراهيم خالد احمد الاجهوري', 'entities': {'PERSON': [{'start': 0, 'end': 3}]}}
{'text': 'عبدالمنعم', 'entities': {}}
{'text': 'دكتور سالم', 'entities': {}}
{'text': 'معاك قمر محمود فهمي محمد شعبان', 'entities': {}}
{'text': 'ثريا عبد الفتاح عزوز علام', 'entities': {'PERSON': [{'start': 0, 'end': 3}]}}
{'text': 'سيد', 'entities': {}}
{'text': 'معاك قمر', 'entities': {}}
{'text': 'احمد عزت احمد على بيكلمك', 'entities': {'PERSON': [{'start': 0, 'end': 0}, {'start': 2, 'end': 2}]}}
{'text': 'نجلاء سيد عبد الحليم منصور', 'entities': {'ORGANIZATION': [{'start': 0, 'end': 0}], 'PERSON': [{'start': 1, 'end': 2}], 'LOCATION': [{'start': 3, 'end': 3}]}}
{'text': 'عطيه', 'entities': {}}
{'text': 'احمد بيكلمك', 'entities': {'PERSON': [{'start': 0, 'end': 0}]}}
{'text': 'بيكلمك رفعت محمد عبدالعزيز شعبان', 'entities': {}}
{'text

In [ ]:
df.shape

(133, 1)